# Bulgarian phoneme FastSpeech2 inference

Run this notebook in Colab from the same Google Drive directory used by `Bulgarian_Phoneme_A100_Colab.ipynb`.

It uses the trained A100 checkpoints in `output_prosody_v2`, restores the small phoneme assets archive, restores only the minimal preprocessed metadata needed for inference, and calls the repo's shared Bulgarian phonemization path.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Use the same values as in the A100 training notebook.
REPO_URL = 'https://github.com/YOUR_USER/FastSpeech2.git'
BRANCH = 'phoneme-mfa'
DRIVE_DIR = '/content/drive/MyDrive/fs2_bg_phone'

# Keep False unless repo_commit_prosody_v2.txt already points to a commit
# that contains this inference notebook and tools/infer_bulgarian.py.
PIN_TO_A100_COMMIT = False

# If None, the notebook uses the latest checkpoint from output_prosody_v2/ckpt/Bulgarian.
RESTORE_STEP = None

TEXT = 'Аз мога да говоря на български език и литература! Вижте колко добре говоря вече.'

# Larger duration_control => slower speech. 1.05-1.25 is usually a sane range for this model.
DURATION_CONTROL = 1.15
PITCH_CONTROL = 1.0
ENERGY_CONTROL = 1.0

# Keep this True if you want unknown words to be phonemized by MFA G2P.
# If all words are already in lexicon/g2p_cache, inference may work without it.
INSTALL_MFA_G2P = True

In [ ]:
import os, shutil, subprocess
from pathlib import Path

os.chdir('/content')
shutil.rmtree('FastSpeech2', ignore_errors=True)
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, 'FastSpeech2'], check=True)
os.chdir('/content/FastSpeech2')

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
commit_file = f'{DRIVE_DIR}/repo_commit_prosody_v2.txt'
training_commit = open(commit_file).read().strip() if os.path.isfile(commit_file) else None
if PIN_TO_A100_COMMIT and training_commit:
    expected = training_commit
    if commit != expected:
        subprocess.run(['git', 'checkout', '--detach', expected], check=True)
        commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
    assert commit == expected, f'Inference must use {expected}, got {commit}'
elif not training_commit:
    open(commit_file, 'w').write(commit + '\n')
    training_commit = commit

print('repo commit used for inference:', commit)
print('A100 training commit recorded in Drive:', training_commit)

In [ ]:
import os, subprocess, sys, zipfile

packages = ['librosa', 'pyworld', 'scikit-learn', 'matplotlib', 'soundfile', 'PyYAML', 'tqdm']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

if not os.path.isfile('hifigan/generator_universal.pth.tar'):
    with zipfile.ZipFile('hifigan/generator_universal.pth.tar.zip') as zf:
        zf.extractall('hifigan')
assert os.path.isfile('hifigan/generator_universal.pth.tar'), 'HiFi-GAN vocoder missing'
print('Python dependencies and vocoder OK')

In [ ]:
import os, shutil, zipfile
from pathlib import Path

drive_dir = Path(DRIVE_DIR)
assets_zip = drive_dir / 'phoneme_assets.zip'
feature_zip = drive_dir / 'preprocessed_Bulgarian_prosody_v2.zip'
ckpt_dir = drive_dir / 'output_prosody_v2' / 'ckpt' / 'Bulgarian'
result_dir = drive_dir / 'inference_results'
result_dir.mkdir(parents=True, exist_ok=True)

assert assets_zip.is_file(), f'Missing {assets_zip}'
assert feature_zip.is_file(), f'Missing {feature_zip}'
assert ckpt_dir.is_dir(), f'Missing checkpoint directory {ckpt_dir}'

# Runtime lexicon, phone inventory sidecar, punctuation metadata, validation marker.
shutil.unpack_archive(str(assets_zip), '.')

# For inference we do not need all mel/duration/pitch/energy arrays; stats.json is enough
# for pitch/energy denormalization in the plot, and g2p_cache.json is useful if present.
shutil.rmtree('preprocessed_data/Bulgarian', ignore_errors=True)
wanted_suffixes = {
    'Bulgarian/stats.json',
    'Bulgarian/speakers.json',
    'Bulgarian/linguistic_abi.json',
    'Bulgarian/g2p_cache.json',
}
with zipfile.ZipFile(feature_zip) as zf:
    names = zf.namelist()
    extracted = []
    for member in names:
        normalized = member.strip('/')
        if any(normalized.endswith(suffix) for suffix in wanted_suffixes):
            if normalized.startswith('preprocessed_data/'):
                zf.extract(member, '.')
            else:
                zf.extract(member, 'preprocessed_data')
            extracted.append(member)

stats = Path('preprocessed_data/Bulgarian/stats.json')
assert stats.is_file(), 'Could not restore preprocessed_data/Bulgarian/stats.json'

drive_g2p_cache = drive_dir / 'g2p_cache_prosody_v2.json'
local_g2p_cache = Path('preprocessed_data/Bulgarian/g2p_cache.json')
if drive_g2p_cache.is_file():
    local_g2p_cache.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_g2p_cache, local_g2p_cache)
    print('restored Drive G2P cache:', drive_g2p_cache)

checkpoints = sorted(ckpt_dir.glob('*.pth.tar'), key=lambda p: int(p.stem.split('.')[0]) if p.stem.split('.')[0].isdigit() else -1)
assert checkpoints, f'No checkpoints found in {ckpt_dir}'
print('restored inference assets:')
print(' ', stats)
print('latest checkpoint:', checkpoints[-1])
print('result dir:', result_dir)

In [ ]:
import os, subprocess
from pathlib import Path

MAMBA_ROOT = '/content/micromamba'
MFA_ROOT_DIR = '/content/mfa_root'
MICROMAMBA = '/content/bin/micromamba'
MFA_CMD = f'{MAMBA_ROOT}/envs/mfa/bin/mfa'
MFA_BIN = str(Path(MFA_CMD).parent)

if INSTALL_MFA_G2P:
    os.environ['MAMBA_ROOT_PREFIX'] = MAMBA_ROOT
    os.environ['MFA_ROOT_DIR'] = MFA_ROOT_DIR
    os.environ['PATH'] = MFA_BIN + os.pathsep + os.environ['PATH']

    if not Path(MICROMAMBA).is_file():
        subprocess.run(
            ['bash', '-lc', 'curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content bin/micromamba'],
            check=True,
        )

    if not Path(MFA_CMD).is_file():
        subprocess.run([
            MICROMAMBA, 'create', '-y', '-n', 'mfa', '-c', 'conda-forge',
            'python=3.11', 'montreal-forced-aligner', 'openfst', 'pynini'
        ], check=True)
    elif not Path(MFA_BIN, 'fstcompile').is_file():
        subprocess.run([
            MICROMAMBA, 'install', '-y', '-n', 'mfa', '-c', 'conda-forge',
            'montreal-forced-aligner', 'openfst', 'pynini'
        ], check=True)

    env = os.environ.copy()
    env['PATH'] = MFA_BIN + os.pathsep + env['PATH']
    subprocess.run([MFA_CMD, 'model', 'download', 'g2p', 'bulgarian_mfa'], check=True, env=env)
    assert Path(MFA_BIN, 'fstcompile').is_file(), 'OpenFST fstcompile missing from MFA env'
    print('MFA G2P OK:', MFA_CMD)
else:
    MFA_CMD = 'mfa'
    MFA_BIN = ''
    print('MFA G2P install skipped; inference works only when every word is in lexicon/cache.')

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
from IPython.display import Audio, display

ckpt_dir = Path(DRIVE_DIR) / 'output_prosody_v2' / 'ckpt' / 'Bulgarian'
result_dir = Path(DRIVE_DIR) / 'inference_results'

cmd = [
    sys.executable, 'tools/infer_bulgarian.py',
    '--text', TEXT,
    '--ckpt-dir', str(ckpt_dir),
    '--result-dir', str(result_dir),
    '--duration-control', str(DURATION_CONTROL),
    '--pitch-control', str(PITCH_CONTROL),
    '--energy-control', str(ENERGY_CONTROL),
    '--mfa-cmd', MFA_CMD,
]
if RESTORE_STEP is not None:
    cmd += ['--restore-step', str(RESTORE_STEP)]
if INSTALL_MFA_G2P:
    cmd += ['--mfa-bin', MFA_BIN, '--mfa-root-dir', MFA_ROOT_DIR, '--mamba-root-prefix', MAMBA_ROOT]

env = os.environ.copy()
if INSTALL_MFA_G2P:
    env['PATH'] = MFA_BIN + os.pathsep + env['PATH']
    env['MFA_ROOT_DIR'] = MFA_ROOT_DIR
    env['MAMBA_ROOT_PREFIX'] = MAMBA_ROOT

result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
print(result.stdout)
result.check_returncode()

wav = max(result_dir.glob('*.wav'), key=lambda p: p.stat().st_mtime)
png = wav.with_suffix('.png')
print('latest wav:', wav)
if png.is_file():
    print('latest plot:', png)

local_g2p_cache = Path('preprocessed_data/Bulgarian/g2p_cache.json')
drive_g2p_cache = Path(DRIVE_DIR) / 'g2p_cache_prosody_v2.json'
if local_g2p_cache.is_file():
    shutil.copy2(local_g2p_cache, drive_g2p_cache)
    print('saved G2P cache:', drive_g2p_cache)
display(Audio(str(wav), rate=22050))

To synthesize another sentence, change `TEXT` and rerun only the last cell. If you changed the checkpoint, also update `RESTORE_STEP`.